<a href="https://colab.research.google.com/github/NayraSousa/mrfi-teste/blob/dev/resnet18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install mrfi
!pip install torch
!pip install torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [45]:
from mrfi import MRFI, EasyConfig
import torch
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from mrfi.experiment import Acc_experiment, Acc_golden
import math
import random
import csv
from torchvision.models import resnet18
import os


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [46]:
class ResNet18ForMNIST(nn.Module):
    def __init__(self, trained=False, num_classes=10):
        super(ResNet18ForMNIST, self).__init__()

        self.backbone = resnet18(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(512, num_classes)

        if trained:
          self.load_state_dict(torch.load('/content/drive/MyDrive/LeNet/train/resnet18_mnist.pth'))


    def forward(self, x):
        return self.backbone(x)

def train(model, epochs=3):
  model.to(device)

  criterion = nn.CrossEntropyLoss()
  optimizer = optim.Adam(model.parameters(), lr=0.001)

  model.train()

  for epoch in range(epochs):
    running_loss = 0
    batch_size = 100
    for i, data in trainloader:
      inputs, labels = i.to(device), data.to(device)

      optimizer.zero_grad()

      outputs = model(inputs)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(trainloader):.4f}")
  torch.save(model.state_dict(), '/content/drive/MyDrive/LeNet/train/resnet18_mnist.pth')
  return model

def test(model):
  model = model.to(device)
  model.load_state_dict(torch.load('/content/drive/MyDrive/LeNet/train/resnet18_mnist.pth'))
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
      for images, labels in testloader:
          images, labels = images.to(device), labels.to(device)
          outputs = model(images)
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()
          break

      print('Accuracy of the network on the 10000 test images: %d %%' % (
        100 * correct / total))

def get_mnist(batch_size=64):
  transform = transforms.Compose(
      [transforms.Resize((224, 224)),
        transforms.ToTensor(),
       transforms.Normalize((0.1307,), (0.3081,))]
  )

  trainset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=True, download=True, transform=transform
  )
  testset = torchvision.datasets.MNIST(
      root='/content/drive/MyDrive/LeNet/data', train=False, download=True, transform=transform
  )

  trainloader = torch.utils.data.DataLoader(
      trainset, batch_size=batch_size, shuffle=True
  )
  testloader = torch.utils.data.DataLoader(
      testset, batch_size=batch_size, shuffle=False
  )

  return trainloader, testloader, testset

def load_dataset(dataset='mnist', batch_size=64):

  if dataset.lower()=='mnist':
    train, test, testset = get_mnist()

    return train, test, testset

In [65]:
def calculates_number_positions(e, N, t, p=0.5):

  denominador = 1 + (e ** 2) * ((N-1) / ((t**2*p*(1-p))))
  N_inj = N / denominador

  return N_inj

def return_abs_value(model, layer_name, position):

  if layer_name == 'conv1':
    flattened_weights = model.backbone.conv1.weight.data.flatten()
    magnitude = torch.abs(flattened_weights[position]).item()

  if layer_name == 'fc':
    flattened_weights = model.backbone.fc.weight.data.flatten()
    magnitude = torch.abs(flattened_weights[position]).item()

  return magnitude

def return_fi_model(model, layer_name, position, n):

  if layer_name == 'conv1':
    config_str = f"""
                    faultinject:
                      - type: weights
                        name: [weight]
                        quantization:
                          method: SymmericQuantization
                          bit_width: 8
                          dynamic_range: auto
                        selector:
                          method: FixPosition
                          position: {position}
                        error_mode:
                          method: IntFixedBitFlip
                          bit_width: 8
                          bit: {n}
                        module_name: conv1
                    """
  if layer_name == 'fc3':
    config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPosition
                            position: {position}
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: {n}
                          module_name: fc3
                          """
  econfig = EasyConfig.load_string(config_str)
  fi_model = MRFI(model.eval(), econfig)
  fi_model.to(device)

  return fi_model, econfig

def return_calculation_times(model, layer_name, input_size=28):
  if layer_name == 'conv1':
    OFW = ((input_size + 2*model.backbone.conv1.padding[0] - model.backbone.conv1.kernel_size[0]) // model.backbone.conv1.stride[0]) + 1
    OFH = OFW
    CT_i = OFW * OFH

    return CT_i

  if layer_name == 'fc3':
    return 1

def return_gradient_value(model, layer_name):

  if layer_name == 'conv1':
    return model.backbone.conv1.weight.grad.data.flatten()

  if layer_name == 'fc3':
    return model.backbone.fc.weight.grad.data.flatten()

In [56]:
grad_lenet.backbone.conv1.weight.grad.data.flatten()

tensor([-5.3020e-05, -4.9533e-05, -5.9744e-05,  ..., -1.4080e-04,
        -1.6726e-04, -1.9223e-04], device='cuda:0')

In [48]:
def evaluate_with_injection(fi_model, loader):
    fi_model.eval()
    correct_gold = 0
    correct_inj  = 0
    total = 0

    fi_model.observers_reset()

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        total += labels.size(0)

        with fi_model.golden_run():
            out_golden = fi_model(images)
            _, pred_g = out_golden.max(dim=1)
            correct_gold += (pred_g == labels).sum().item()

        out_inject = fi_model(images)
        _, pred_f = out_inject.max(dim=1)
        correct_inj += (pred_f == labels).sum().item()

    acc_golden = correct_gold / total
    acc_inject = correct_inj  / total

    return acc_golden, acc_inject

In [42]:
trainloader, testloader,testset = load_dataset()

resnet = ResNet18ForMNIST()
grad_resnet = train(resnet)
test(resnet)

for name, param in resnet.named_parameters():
    print(f"Camada: {name}")
    print(f"Parâmetros: {param.count_nonzero()}")
    print("-" * 50)

Epoch 1/3, Loss: 0.0997
Epoch 2/3, Loss: 0.0403
Epoch 3/3, Loss: 0.0330
Accuracy of the network on the 10000 test images: 98 %
Camada: backbone.conv1.weight
Parâmetros: 3136
--------------------------------------------------
Camada: backbone.bn1.weight
Parâmetros: 64
--------------------------------------------------
Camada: backbone.bn1.bias
Parâmetros: 64
--------------------------------------------------
Camada: backbone.layer1.0.conv1.weight
Parâmetros: 36864
--------------------------------------------------
Camada: backbone.layer1.0.bn1.weight
Parâmetros: 64
--------------------------------------------------
Camada: backbone.layer1.0.bn1.bias
Parâmetros: 64
--------------------------------------------------
Camada: backbone.layer1.0.conv2.weight
Parâmetros: 36864
--------------------------------------------------
Camada: backbone.layer1.0.bn2.weight
Parâmetros: 64
--------------------------------------------------
Camada: backbone.layer1.0.bn2.bias
Parâmetros: 64
----------------

In [66]:
n_conv1 = calculates_number_positions(0.05, 3136, 2.60)
n_fc3 = calculates_number_positions(0.05, 5120, 2.60)

pos_conv1 = random.sample(range(3136), int(n_conv1))
print(pos_conv1)

pos_fc3 = random.sample(range(5120), int(n_fc3))
print(pos_fc3)

[594, 2885, 2030, 2214, 401, 3052, 48, 666, 2044, 395, 505, 812, 494, 1095, 22, 487, 2221, 2948, 43, 820, 1804, 805, 2035, 850, 1446, 1000, 2156, 2218, 1729, 1313, 1878, 1180, 987, 1996, 2759, 2315, 544, 443, 2482, 741, 1767, 678, 2701, 490, 2042, 3003, 1680, 3050, 1765, 1228, 2939, 1497, 555, 1594, 513, 648, 1432, 943, 1043, 1628, 2783, 261, 720, 1557, 410, 3114, 671, 2938, 239, 93, 1076, 2199, 577, 2243, 188, 1359, 1838, 212, 946, 2276, 381, 2481, 504, 1128, 2004, 2642, 3084, 2436, 1678, 1887, 1287, 2455, 511, 2382, 2447, 2099, 3071, 1216, 323, 451, 1474, 187, 2829, 2154, 2541, 526, 2681, 1397, 2421, 2539, 112, 2588, 2264, 2088, 2685, 257, 1876, 1958, 269, 1643, 536, 1526, 39, 294, 2463, 906, 2477, 2354, 1562, 907, 2559, 2295, 2349, 1932, 740, 2858, 2984, 2503, 2771, 372, 1017, 153, 139, 2598, 1623, 1268, 1175, 1570, 2402, 2619, 179, 1787, 2300, 3133, 612, 927, 882, 253, 1961, 1424, 1739, 463, 2348, 3014, 560, 977, 1470, 808, 1455, 3115, 183, 600, 2875, 1666, 595, 1763, 158, 2659, 21

In [63]:
bits = [6, 7]
layers = ['conv1', 'fc3']
metadata = []

CT_conv1 = return_calculation_times(resnet, layers[0])
CT_fc3 = return_calculation_times(resnet, layers[1])

gradient_conv1 = return_gradient_value(grad_resnet, layers[0])
gradient_fc3 = return_gradient_value(grad_resnet, layers[1])

In [ ]:
for n in bits:
  for name in layers:

    if name == 'conv1':
      for position in pos_conv1:

        resnet = ResNet18ForMNIST(trained=True)
        magnitude = return_abs_value(resnet, name, position)

        fi_model, econfig = return_fi_model(resnet, name, position, n)
        acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

        metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
                         'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
                         'vulnerability': acc_g-acc_f, 'calculation_times': CT_conv1,
                         'gradient': gradient_conv1[position].item()})

    if name == 'fc3':
      for position in pos_fc3:

        resnet = ResNet18ForMNIST(trained=True)
        magnitude = return_abs_value(resnet, name, position)

        fi_model, econfig = return_fi_model(resnet, name, position, n)
        acc_g, acc_f = evaluate_with_injection(fi_model, testloader)

        metadata.append({f'layer_name': name, 'bit': econfig.faultinject[0]['error_mode']['args']['bit'],
                         'parameter': econfig.faultinject[0]['selector']['args']['position'], 'magnitude': magnitude,
                         'vulnerability': acc_g-acc_f, 'calculation_times': CT_fc3,
                         'gradient': gradient_fc3[position].item()})

In [ ]:
file_path = '/content/drive/MyDrive/LeNet/analise_resnet18.csv'
write_header = not os.path.exists(file_path)

with open(file_path, 'a', newline='', encoding='utf-8') as file:
    escritor = csv.DictWriter(file, fieldnames=metadata[0].keys())

    if write_header:
        escritor.writeheader()
    escritor.writerows(metadata)

In [ ]:
lenet = ResNet18ForMNIST(trai ned=True)
config_str = f"""
                    faultinject:
                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPositions
                            positions: [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 133, 123, 118, 112, 20, 127, 40, 128, 117, 7, 116]
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: 7
                          module_name: conv1

                        - type: weights
                          name: [weight]
                          quantization:
                            method: SymmericQuantization
                            bit_width: 8
                            dynamic_range: auto
                          selector:
                            method: FixPositions
                            positions: [812, 826, 818, 788, 784, 796, 816, 444, 596, 74, 776, 653, 829, 493, 595, 438, 498, 420, 660, 503, 483, 448, 442, 467, 227]
                          error_mode:
                            method: IntFixedBitFlip
                            bit_width: 8
                            bit: 7
                          module_name: fc3
                          """
econfig = EasyConfig.load_string(config_str)
fi_model = MRFI(lenet.eval(), econfig)
fi_model.to(device)

acc_g, acc_f = evaluate_with_injection(fi_model, testloader)
acc_g, acc_f

(0.9863, 0.2673)

# CONV1

0.9795 [11]

0.9838 [11, 103]

0.9797 [11, 103, 5]

0.9852 [11, 103, 5, 108]

0.9754 [11, 103, 5, 108, 1]

0.9476 [11, 103, 5, 108, 1, 104]

0.973 [11, 103, 5, 108, 1, 104, 107]

0.9084 [11, 103, 5, 108, 1, 104, 107, 10]

0.8319 [11, 103, 5, 108, 1, 104, 107, 10, 6]

0.892 [11, 103, 5, 108, 1, 104, 107, 10, 6, 21]

## Tirando os gradientes negativos em ordem decrescente:

0.8272 [11, 103, 5, 1, 104, 107, 10, 6, 21]

0.7689 [11, 103, 5, 1, 104, 10, 6, 21]

0.8499 [11, 103, 5, 1, 10, 6, 21]

0.9125 [11, 5, 1, 10, 6, 21]

## Botando em ordem crescente de gradient:

0.9125 [5, 10, 1, 6, 21, 11]

## Botando em ordem decrescente de vulnerabilidade:

0.9125 [11, 5, 1, 10, 6, 21]

## Menor acurácia:

Foi dado pelas seguintes posições: [11, 103, 5, 1, 104, 10, 6, 21]. Agora vou testar com essas posições e o restante das 15 mais vulneravéis.

## Adicionando as posições a partir da 11° mais vulnerável:

0.7023 [11, 103, 5, 1, 104, 10, 6, 21, 124]

0.7773 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109]

0.7444  [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0]

0.6529 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0, 119]

0.6246 [11, 103, 5, 1, 104, 10, 6, 21, 124, 109, 0, 119, 15]

## Tirando os gradientes negativos em ordem decrescente:

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]

0.6446 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 15]

0.7108  [11, 103, 5, 1, 104, 10, 6, 21, 0, 15]

## Botando em ordem crescente de gradient:

0.8598 [15, 5, 10, 0, 1, 6, 21, 11]

## Botando em ordem crescente de vulnerabilidade:

0.8598 [11, 5, 1, 10, 6, 21, 0, 15]

## Testando a partir do 16° bit mais vulneravel:

0.5333 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 114, 133]

0.5441  [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 114]

0.501 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111]

0.5411 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106]

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]

## Tirando os gradientes negativos em ordem decrescente:

0.4951 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111, 133]

0.501  [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106, 111]

0.5411 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15, 106]

0.5582 [11, 103, 5, 1, 104, 10, 6, 21, 124, 0, 119, 15]